## No load added, just 100 MWh at Bus 3

In [ ]:
import cvxpy as cp

# Variables
Pg1 = cp.Variable()
Pg2 = cp.Variable()
Snom = cp.Variable()

# Voltage angles
t1 = 0
t2 = cp.Variable()

# Parameters
b12 = 10

PL1 = 0
PL2 = 200

# Line flow
F12 = (t1 - t2) * b12

# Objective
objective = cp.Minimize(
    1000*Pg1 +
    12000*Pg2 +
    6100*Snom
)

constraints = [

    # Generator limits
    Pg1 >= 0,
    Pg1 <= 120,

    Pg2 >= 0,
    Pg2 <= 100,

    # Transmission capacity
    Snom >= 75,
    Snom <= 150,

    # Power balance
    Pg1 - PL1 == F12,
    Pg2 - PL2 == -F12,

    # Line limits
    F12 <= Snom,
    F12 >= -Snom
]

# Solve
prob = cp.Problem(objective, constraints)
prob.solve()

print("Status =", prob.status)

if prob.status == "optimal":
    print("Cost =", prob.value)
    
    print("Pg1 =", Pg1.value)
    print("Pg2 =", Pg2.value)
    print("Snom =", Snom.value)
    print("Flow =", F12.value)

Status = optimal
Cost = 1812000.0009949077
Pg1 = 119.99999988061248
Pg2 = 80.00000011938748
Snom = 119.99999994781075
Flow = 119.99999988061246


In [9]:
print("\nGeneration")
print("P1 =", Pg1.value)
print("P2 =", Pg2.value)


print("\nVoltage Angles")
print("t2 =", t2.value)



Generation
P1 = 119.99999999944089
P2 = 80.00000000055907

Voltage Angles
t2 = -11.999999999944091


In [10]:
# Get LMPs to see what each gen gets paid at...

LMP1 = -constraints[6].dual_value
LMP2 = -constraints[7].dual_value

In [11]:
print("LMP Bus 1:", LMP1)
print("LMP Bus 2:", LMP2)

LMP Bus 1: 5900.000003705085
LMP Bus 2: 12000.000004706397


In [5]:
congestion_rent = (LMP2 - LMP1) * F12.value

In [6]:
congestion_rent

np.float64(732000.0064420736)

In [5]:
import pypsa

n = pypsa.Network()
n.set_snapshots([0])  # single timestep

# Two buses
n.add("Bus", "BusA")
n.add("Bus", "BusB")

# Generator at Bus A
n.add("Generator", "GenA",
      bus="BusA",
      control = 'Slack',
      p_nom=120,
      marginal_cost=00)

#load at bus A
n.add("Load", "LoadA",
      bus="BusA",
      p_set=2000,
      )


# Load at Bus B
n.add("Load", "LoadB",
      bus="BusB",
      p_set=200)

# generatror as bus b
n.add("Generator", "GenB",
      bus="BusB",
      p_nom=100,
      marginal_cost=12000)

# Extendable line
n.add("Line", "Line1",
      bus0="BusA",
      bus1="BusB",
      s_nom=0,              # start from 0
      s_nom_min=0,          # minimum capacity
      s_nom_max=300,        # maximum capacity
      s_nom_extendable=True,
      capital_cost=6100,    # Rs per MVA — annualized
      x=0.1)                # reactance for DC flow

n.optimize()

# Results
print(n.lines.s_nom_opt)        # → 150 MW
print(n.buses_t.marginal_price) # → LMP at each bus
print(n.lines_t.p0)             # → power flow

/var/folders/zk/lh41nc414yl79qyknsg_d3kr0000gn/T/ipykernel_2045/3915261362.py:46: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n.optimize()
Index(['BusA', 'BusB'], dtype='str', name='name')
Index(['Line1'], dtype='str', name='name')
Index(['Line1'], dtype='str', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.02s
Status: warning
Termination condition: infeasible
Solution: 0 primals, 0 duals
Objective: nan
Solver model: available
Solver message: Infeasible



Running HiGHS 1.14.0 (git hash: n/a): Copyright (c) 2026 under MIT licence terms
LP linopy-problem-ticr7_c_ has 10 rows; 4 cols; 14 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [6e+03, 1e+04]
  Bound   [0e+00, 0e+00]
  RHS     [1e+02, 2e+03]
Presolving model
Problem status detected on presolve: Infeasible
Model name          : linopy-problem-ticr7_c_
Model status        : Infeasible
Objective value     :  0.0000000000e+00
HiGHS run time      :          0.00
name
Line1    0.0
Name: s_nom_opt, dtype: float64
Empty DataFrame
Columns: []
Index: [0]
Empty DataFrame
Columns: []
Index: [0]


In [7]:
n.buses_t

{'v_mag_pu_set': Empty DataFrame
 Columns: []
 Index: [0],
 'p': Empty DataFrame
 Columns: []
 Index: [0],
 'q': Empty DataFrame
 Columns: []
 Index: [0],
 'v_mag_pu': Empty DataFrame
 Columns: []
 Index: [0],
 'v_ang': Empty DataFrame
 Columns: []
 Index: [0],
 'marginal_price': Empty DataFrame
 Columns: []
 Index: [0]}

In [45]:
import cvxpy as cp
from numpy import round
# =========================================================
# VARIABLES
# =========================================================

Pg1 = cp.Variable(name="Pg1")
Pg2 = cp.Variable(name="Pg2")

Snom = cp.Variable(name="Snom")

t2 = cp.Variable(name="theta2")

# =========================================================
# PARAMETERS
# =========================================================

t1 = 0

b12 = 10

PL1 = 0
PL2 = 200

# =========================================================
# LINE FLOW
# =========================================================

F12 = (t1 - t2) * b12

# =========================================================
# OBJECTIVE FUNCTION
# =========================================================

objective = cp.Minimize(
    1000 * Pg1 +
    12000 * Pg2 +
    6100 * Snom
)

# =========================================================
# CONSTRAINTS
# =========================================================

constraints = []

# Generator limits
constraints += [
    Pg1 >= 0,
    Pg1 <= 120,

    Pg2 >= 0,
    Pg2 <= 100
]

# Transmission expansion limits
constraints += [
    Snom >= 75,
    Snom <= 150
]

# Power balance equations
pb1 = Pg1 - PL1 == F12
pb2 = Pg2 - PL2 == -F12

constraints += [pb1, pb2]

# Line capacity constraints
line_upper = F12 <= Snom
line_lower = F12 >= -Snom

constraints += [
    line_upper,
    line_lower
]

# =========================================================
# SOLVE
# =========================================================

prob = cp.Problem(objective, constraints)
prob.solve()

# =========================================================
# RESULTS
# =========================================================

print("\n============================")
print("STATUS")
print("============================")

print(prob.status)

print("\n============================")
print("OPTIMAL COST")
print("============================")

print(prob.value)

print("\n============================")
print("GENERATION")
print("============================")

print("Pg1 =", round(Pg1.value,2), "MW")
print("Pg2 =", round(Pg2.value,2), "MW")

print("\n============================")
print("TRANSMISSION")
print("============================")

print("Snom =", round(Snom.value,2), "MW")
print("Flow =", round(F12.value,2), "MW")

print("\n============================")
print("ANGLES")
print("============================")

print("theta1 =", t1)
print("theta2 =", round(t2.value,4))

# =========================================================
# LMPs
# =========================================================

print("\n============================")
print("LMPs")
print("============================")

# Negative sign because CVXPY convention
LMP1 = -pb1.dual_value
LMP2 = -pb2.dual_value

print("LMP Bus 1 =", round(LMP1,2))
print("LMP Bus 2 =", round(LMP2,2))

# =========================================================
# CONGESTION DUALS
# =========================================================

print("\n============================")
print("LINE CONSTRAINT DUALS")
print("============================")

print("Upper limit dual =", line_upper.dual_value)
print("Lower limit dual =", line_lower.dual_value)

# =========================================================
# CONGESTION RENT
# =========================================================

print("\n============================")
print("CONGESTION RENT")
print("============================")

congestion_rent = (LMP2 - LMP1) * F12.value

print("Congestion Rent =", round(congestion_rent,2))

# =========================================================
# INTERPRETATION
# =========================================================

print("\n============================")
print("INTERPRETATION")
print("============================")

if abs(line_upper.dual_value) > 1e-6:
    print("Line is congested in positive direction.")

if abs(line_lower.dual_value) > 1e-6:
    print("Line is congested in negative direction.")

if abs(LMP1 - LMP2) < 1e-6:
    print("No congestion: LMPs equal.")

else:
    print("Congestion exists: LMP separation observed.")


STATUS
optimal

OPTIMAL COST
1812000.0009949077

GENERATION
Pg1 = 120.0 MW
Pg2 = 80.0 MW

TRANSMISSION
Snom = 120.0 MW
Flow = 120.0 MW

ANGLES
theta1 = 0
theta2 = -12.0

LMPs
LMP Bus 1 = 5900.0
LMP Bus 2 = 12000.0

LINE CONSTRAINT DUALS
Upper limit dual = 6100.000065635947
Lower limit dual = 5.883132817646277e-06

CONGESTION RENT
Congestion Rent = 732000.01

INTERPRETATION
Line is congested in positive direction.
Line is congested in negative direction.
Congestion exists: LMP separation observed.


In [47]:
732000/6100

120.0

In [16]:
import cvxpy as cp

# =========================================================
# VARIABLES
# =========================================================

Pg1 = cp.Variable(name="Pg1")
Pg2 = cp.Variable(name="Pg2")

theta2 = cp.Variable(name="theta2")

# =========================================================
# PARAMETERS
# =========================================================

theta1 = 0          # Slack bus angle
b12 = 1             # Line susceptance

# Loads
PL1 = 0
PL2 = 150

# Generator costs ($/MWh)
c1 = 10
c2 = 50

# Generator limits
Pg1_max = 120
Pg2_max = 100

# Line limit
Fmax = 40

# =========================================================
# LINE FLOW
# =========================================================

F12 = b12 * (theta1 - theta2)

# =========================================================
# OBJECTIVE FUNCTION
# =========================================================

objective = cp.Minimize(
    c1 * Pg1 +
    c2 * Pg2
)

# =========================================================
# CONSTRAINTS
# =========================================================

constraints = []

# Generator limits
constraints += [
    Pg1 >= 0,
    Pg1 <= Pg1_max,

    Pg2 >= 0,
    Pg2 <= Pg2_max
]

# Nodal power balance equations
pb1 = Pg1 - PL1 == F12
pb2 = Pg2 - PL2 == -F12

constraints += [pb1, pb2]

# Line flow limits
line_upper = F12 <= Fmax
line_lower = F12 >= -Fmax

constraints += [
    line_upper,
    line_lower
]

# =========================================================
# SOLVE
# =========================================================

problem = cp.Problem(objective, constraints)
problem.solve()

# =========================================================
# RESULTS
# =========================================================

print("\n============================")
print("OPTIMAL DISPATCH")
print("============================")

print(f"Pg1 = {Pg1.value:.2f} MW")
print(f"Pg2 = {Pg2.value:.2f} MW")

print("\n============================")
print("POWER FLOW")
print("============================")

print(f"Flow 1->2 = {F12.value:.2f} MW")

print("\n============================")
print("LOCATIONAL MARGINAL PRICES")
print("============================")

# Dual values of power balance equations
LMP1 = -pb1.dual_value
LMP2 = -pb2.dual_value

print(f"LMP Bus 1 = ${LMP1:.2f}/MWh")
print(f"LMP Bus 2 = ${LMP2:.2f}/MWh")

print("\n============================")
print("CONGESTION")
print("============================")

print(f"Upper line constraint dual = {line_upper.dual_value:.2f}")
print(f"Lower line constraint dual = {line_lower.dual_value:.2f}")

# Congestion rent
congestion_rent = (LMP2 - LMP1) * F12.value

print(f"\nCongestion Rent = ${congestion_rent:.2f}/h")

print("\n============================")
print("TOTAL COST")
print("============================")

print(f"Total Cost = ${problem.value:.2f}/h")


OPTIMAL DISPATCH


TypeError: unsupported format string passed to NoneType.__format__

In [23]:
Pg1.values

AttributeError: 'Variable' object has no attribute 'values'

In [ ]:


print('\nLoad at bus 1:', PL1, 'MWh. LMP: $', -np.round(LMP1,2), '. Pays: $', round(-PL1*LMP1, 2))
print('Load at bus 2:', PL2, 'MWh. LMP: $', -np.round(LMP2,2),'. Pays: $', round(-PL2*LMP2, 2))
print('Load at bus 3:', PL3, 'MWh. LMP: $', -np.round(LMP3,2),'. Pays: $', round(-PL3*LMP3, 2))

# Any line constraints binding?
# print('\nBinding line constraints?')
# print(constraints[8].dual_value + constraints[9].dual_value > 0.01)

print('\nTotal system operating cost : $', np.round(objective.value,3))

print("\nPg1: ", np.round(abs(P1.value),2), " MWh, gets paid: $", round(-P1.value*LMP1,2))
print("Pg2: ", np.round(abs(P2.value),2), " MWh, gets paid: $", round(-P2.value*LMP2, 2))
print("Pg3: ", np.round(abs(P3.value),2), " MWh, gets paid: $", round(-P3.value*LMP3, 2))
#print("Bus 3: ", PL3, " MWh, load pays: $", round(PL3*LMP3, 3))
#print("Marginal price: $", np.round(MP,2))

print('\n supply meets demand? : ', PL1+PL2+PL3, 'load and ', P1.value+P2.value+P3.value, ' gen')

## Now consider adding load - 70 MWh at Bus 1

In [ ]:
import cvxpy as cp
import numpy as np

P1 = cp.Variable()
P2 = cp.Variable()
P3 = cp.Variable()
t1 = 0 # reference bus
t2 = cp.Variable()
t3 = cp.Variable()

# line susceptance
b12 = 1
b13 = 1
b23 = 1

# loads
PL1 = 70
PL2 = 0
PL3 = 100

objective = cp.Minimize(10*P1 + 20*P2 + 100*P3)

# Generator limits
constraints = [P1 >= 0, P2 >= 0, P3 >=0, # Minimum generation Limits
               P1 <= 1200, P2 <= 1200] # Maximum generation limits

# Power Balance
constraints += [P1-PL1 == (t1-t2)*b12 + (t1-t3)*b13, # power balance at bus 1
               P2-PL2 == (t2-t1)*b12 + (t2-t3)*b23, # power balance at bus 2
               P3-PL3 == (t3-t1)*b13 + (t3-t2)*b23] # power balance at bus 3

# Line flow limits
constraints += [(t1-t3) <= 10, (t3-t1) <= 10] # line limit between bus 1 and 3
    
prob = cp.Problem(objective, constraints)

results = prob.solve()
print(prob.status)

# Get LMPs to see what each gen gets paid at...
LMP1 = constraints[5].dual_value
LMP2 = constraints[6].dual_value
LMP3 = constraints[7].dual_value


print('\nLoad at bus 1:', PL1, 'MWh. LMP: $', -np.round(LMP1,2), '. Pays: $', round(-PL1*LMP1, 2))
print('Load at bus 2:', PL2, 'MWh. LMP: $', -np.round(LMP2,2),'. Pays: $', round(-PL2*LMP2, 2))
print('Load at bus 3:', PL3, 'MWh. LMP: $', -np.round(LMP3,2),'. Pays: $', round(-PL3*LMP3, 2))

# Any line constraints binding?
# print('\nBinding line constraints?')
# print(constraints[8].dual_value + constraints[9].dual_value > 0.01)

print('\nTotal system operating cost : $', np.round(objective.value,3))

print("\nPg1: ", np.round(abs(P1.value),2), " MWh, gets paid: $", round(-P1.value*LMP1,2))
print("Pg2: ", np.round(abs(P2.value),2), " MWh, gets paid: $", round(-P2.value*LMP2, 2))
print("Pg3: ", np.round(abs(P3.value),2), " MWh, gets paid: $", round(-P3.value*LMP3, 2))
#print("Bus 3: ", PL3, " MWh, load pays: $", round(PL3*LMP3, 3))
#print("Marginal price: $", np.round(MP,2))

print('\n supply meets demand? : ', PL1+PL2+PL3, 'load and ', P1.value+P2.value+P3.value, ' gen')

optimal

Load at bus 1: 70 MWh. LMP: $ 4.4 . Pays: $ 308.3
Load at bus 2: 0 MWh. LMP: $ 20.0 . Pays: $ -0.0
Load at bus 3: 100 MWh. LMP: $ 35.6 . Pays: $ 3559.57

Total system operating cost : $ 3400.0

Pg1:  0.0  MWh, gets paid: $ 0.0
Pg2:  170.0  MWh, gets paid: $ 3400.0
Pg3:  0.0  MWh, gets paid: $ -0.0

 supply meets demand? :  170 load and  169.99999999999048  gen


In [23]:
import pandas as pd

In [40]:
data = pd.read_excel('Hydro_Generation.xlsx')

In [41]:
data = pd.read_excel('Hydro_Generation.xlsx')# 1. Your clean setup (Keep this exactly as you wrote it)
data = data.drop(['Month','No. of Days (A)'], axis=1)
data.Date = pd.to_datetime(data.Date)
data = data.set_index('Date')
data = data.sort_index()

# 2. Your target hourly start and end bounds
start_target = "2020-01-15 12:00:00"
end_target = "2021-12-16 23:00:00" 

# 3. Your hourly target index
target_index = pd.date_range(start=start_target, end=end_target, freq="h")

# ==========================================
# FIX: Change .asfreq() to .ffill() to populate the hours
# ==========================================
# 4 & 5. Resample, fill the gaps, and snap to your target index
df_final = data.resample("h").ffill().reindex(target_index)

print(df_final)


                     Samuktya  Makargard  Chino Hydro  Bagmati Khola  \
2020-01-15 12:00:00      0.75       0.75         0.33           7.06   
2020-01-15 13:00:00      0.75       0.75         0.33           7.06   
2020-01-15 14:00:00      0.75       0.75         0.33           7.06   
2020-01-15 15:00:00      0.75       0.75         0.33           7.06   
2020-01-15 16:00:00      0.75       0.75         0.33           7.06   
...                       ...        ...          ...            ...   
2021-12-16 19:00:00       NaN        NaN          NaN            NaN   
2021-12-16 20:00:00       NaN        NaN          NaN            NaN   
2021-12-16 21:00:00       NaN        NaN          NaN            NaN   
2021-12-16 22:00:00       NaN        NaN          NaN            NaN   
2021-12-16 23:00:00       NaN        NaN          NaN            NaN   

                     Maya Khola  Halesi Hydro  
2020-01-15 12:00:00        0.98          1.97  
2020-01-15 13:00:00        0.98        

In [34]:
data

,Samuktya,Makargard,Chino Hydro,Bagmati Khola,Maya Khola,Halesi Hydro
Date,,,,,,
2020-01-15,0.75,0.75,0.330,7.06,0.98,1.97
2020-02-13,0.76,0.76,0.310,6.36,0.82,1.79
2020-03-15,0.79,0.79,0.330,5.66,0.89,1.81
2020-04-14,1.01,1.01,0.520,8.15,1.59,3.84
2020-05-15,1.35,1.35,0.890,16.77,3.15,6.10
2020-06-15,1.35,1.35,1.065,16.77,3.15,6.10
2020-07-17,1.35,1.35,1.065,16.77,3.15,6.10
2020-08-17,1.35,1.35,1.065,16.77,3.15,6.10
2020-09-17,1.35,1.35,1.065,16.77,3.15,6.10


                     Samuktya  Makargard  Chino Hydro  Bagmati Khola  \
2020-01-15 12:00:00       NaN        NaN          NaN            NaN   
2020-01-15 13:00:00       NaN        NaN          NaN            NaN   
2020-01-15 14:00:00       NaN        NaN          NaN            NaN   
2020-01-15 15:00:00       NaN        NaN          NaN            NaN   
2020-01-15 16:00:00       NaN        NaN          NaN            NaN   
...                       ...        ...          ...            ...   
2021-12-16 19:00:00       NaN        NaN          NaN            NaN   
2021-12-16 20:00:00       NaN        NaN          NaN            NaN   
2021-12-16 21:00:00       NaN        NaN          NaN            NaN   
2021-12-16 22:00:00       NaN        NaN          NaN            NaN   
2021-12-16 23:00:00       NaN        NaN          NaN            NaN   

                     Maya Khola  Halesi Hydro  
2020-01-15 12:00:00         NaN           NaN  
2020-01-15 13:00:00         NaN        